# mse-reconstruction-loss — ex1: scalar mse loss + side-by-side reconstructions

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `mse-reconstruction-loss`. Running the final beacon cell reports progress against the `Generative: MSE reconstruction loss` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: MSE reconstruction loss` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mse-reconstruction-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mse-reconstruction-loss"
DD_SUBTOPIC = "Generative: MSE reconstruction loss"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## MSE reconstruction loss — quick refresher

Autoencoders train against `loss = F.mse_loss(decoded, original)`. The default `reduction='mean'` averages over EVERY element — batch, channel, height, width — giving a single scalar.

**Three reductions to know.**
- `reduction='mean'` (default) — scalar; mean over `B*C*H*W` elements.
- `reduction='sum'` — scalar; sum over all elements (B times bigger than mean for fixed batch).
- `reduction='none'` — per-element `(B,C,H,W)` tensor; you decide how to reduce.

**Per-sample loss.** When you want one loss number per batch element (e.g. to weight some images more heavily): use `reduction='none'`, then `.mean(dim=[1,2,3])` to average within each sample. This gives you a `(B,)` vector.

**Argument order.** `F.mse_loss(input, target)` — both arguments are symmetric (MSE is symmetric in its inputs), so the order doesn't change the value. But by convention `input` is the model output, `target` is the ground truth.

### Exercise 1 — scalar mse loss + side-by-side reconstructions

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `F.mse_loss` with the correct `reduction` arg to compute (a) the default scalar mean over all elements and (b) a `(B,)` per-sample loss vector.
> Keywords: mse, reconstruction, F.mse_loss, visualization
> ```

**KCs targeted:** `mse-default-reduction-mean`, `mse-per-sample-via-none`

Implement `ex1_mse_losses(original, decoded)`. Two reductions, one function call each:

1. `original` and `decoded` both have shape `(B, 1, 28, 28)` — a minibatch of MNIST images.
2. Compute `scalar_loss = F.mse_loss(decoded, original)` — uses the default `reduction='mean'`, returns a 0-D scalar.
3. Compute `per_sample_loss` by calling `F.mse_loss` with `reduction='none'` to get a `(B, 1, 28, 28)` per-element tensor, then averaging over the last three axes to collapse it to `(B,)`.
4. Return a tuple `(scalar_loss, per_sample_loss)`.

Input: `original`, `decoded` — `(B, 1, 28, 28)` float tensors.
Output: tuple of `(scalar 0-D tensor, (B,) tensor)`.

The visualization renders an originals row and a corrupted-reconstructions row side by side, with the per-sample MSE annotated under each pair — you can see which samples the 'decoder' got right vs wrong.

In [ ]:
def ex1_mse_losses(original: Tensor, decoded: Tensor) -> tuple[Tensor, Tensor]:
    import torch.nn.functional as F
    scalar = F.mse_loss(decoded, original)                          # reduction='mean' default
    per_elem = F.mse_loss(decoded, original, reduction='none')      # (B, 1, 28, 28)
    per_sample = per_elem.mean(dim=[1, 2, 3])                       # (B,)
    return scalar, per_sample


<details><summary>Solution</summary>

```python
def ex1_mse_losses(original: Tensor, decoded: Tensor) -> tuple[Tensor, Tensor]:
    import torch.nn.functional as F
    scalar = F.mse_loss(decoded, original)                          # reduction='mean' default
    per_elem = F.mse_loss(decoded, original, reduction='none')      # (B, 1, 28, 28)
    per_sample = per_elem.mean(dim=[1, 2, 3])                       # (B,)
    return scalar, per_sample
```

**Why `reduction='none'` then `.mean(dim=...)`.** This is the only way to get a per-sample loss without re-implementing MSE by hand. `reduction='sum'` gives one scalar; `reduction='mean'` gives one scalar; `reduction='none'` is the one that preserves shape so you can choose your own collapse axes.

**Default scalar vs explicit per-sample.** The default scalar form is what you pass to `loss.backward()` — it's symmetric, differentiable, and averaged. The per-sample form is for LOGGING — you'd never `backward()` directly on a `(B,)` vector (that implicitly sums, which is `reduction='sum'` divided by `H*W` — almost certainly not what you want).

**Argument order.** Convention is `F.mse_loss(input, target)` where `input` is the model output. MSE is symmetric so it doesn't matter mathematically — but matching the convention makes the code readable.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()